# Stage 3 — traffic forensic analysis

This notebook derives traffic-specific evidence from the canonical CDR table and compares known fraudulent and legitimate SIMs. It does not produce a final fraud decision.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED=Path('../outputs/processed/all_naive_12pct_200sims')
OUTPUT=Path('../outputs/traffic/all_naive_12pct_200sims')
OUTPUT.mkdir(parents=True,exist_ok=True)
cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string'},parse_dates=['timestamp'])
features=pd.read_csv('../outputs/statistical/all_naive_12pct_200sims/sim_statistical_features.csv',dtype={'sim_number':'string'})
print('canonical CDRs:',len(cdr),'SIM profiles:',len(features))

canonical CDRs: 86873 SIM profiles: 6992


C:\Users\salai_wciilqo\AppData\Local\Temp\ipykernel_648\792750045.py:8: DtypeWarning: Columns (3: src_imei, 7: dst_imei) have mixed types. Specify dtype option on import or set low_memory=False.
  cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string'},parse_dates=['timestamp'])


In [2]:
voice={'MOC','MTC','I-MOC','I-MTC'}; outgoing={'MOC','I-MOC','MO-SMS'}; incoming={'MTC','I-MTC'}; international={'I-MOC','I-MTC'}

features['international_voice_activity_count']=features['international_activity_count']
features['activity_population_share']=features['activity_count']/features['activity_count'].sum()
features['outgoing_share']=features['outgoing_activity_count']/features['activity_count'].replace(0,np.nan)
features['incoming_share']=features['incoming_activity_count']/features['activity_count'].replace(0,np.nan)
features['contact_per_active_hour']=features['unique_contacts']/features['active_hours'].replace(0,np.nan)
features['duration_per_activity']=features['total_duration']/features['activity_count'].replace(0,np.nan)
features['short_call_count']=0

# Count short voice calls per source SIM as a traffic-specific indicator.
short_source=cdr[cdr.cdr_type.isin(voice)&(cdr.duration<=10)].groupby('src_number').size()
features['short_call_count']=features['sim_number'].map(short_source).fillna(0)
features['short_call_ratio']=features['short_call_count']/features['voice_activity_count'].replace(0,np.nan)

traffic_columns=['activity_count','outgoing_activity_count','incoming_activity_count','outgoing_share','incoming_share','total_duration','duration_per_activity','unique_contacts','contact_per_active_hour','international_activity_count','international_activity_ratio','short_call_count','short_call_ratio','activity_population_share']
features.to_csv(OUTPUT/'traffic_features.csv',index=False)
features.head()

,sim_number,activity_count,total_duration,average_duration,median_duration,maximum_duration,minimum_duration,active_days,active_hours,unique_contacts,...,outgoing_to_incoming_ratio,activity_span_seconds,international_voice_activity_count,activity_population_share,outgoing_share,incoming_share,contact_per_active_hour,duration_per_activity,short_call_count,short_call_ratio
0,+11102241212,1,16,16.0,16.0,16,16,1,1,1,...,0.0,0.0,1,0.00001,0.0,1.0,1.0,16.0,0.0,0.0
1,+11124222602,1,6,6.0,6.0,6,6,1,1,1,...,0.0,0.0,1,0.00001,0.0,1.0,1.0,6.0,0.0,0.0
2,+11185236570,1,415,415.0,415.0,415,415,1,1,1,...,NaN,0.0,1,0.00001,1.0,0.0,1.0,415.0,0.0,0.0
3,+11201660070,1,9,9.0,9.0,9,9,1,1,1,...,0.0,0.0,1,0.00001,0.0,1.0,1.0,9.0,0.0,0.0
4,+11278389544,1,10,10.0,10.0,10,10,1,1,1,...,0.0,0.0,1,0.00001,0.0,1.0,1.0,10.0,0.0,0.0


In [3]:
counts=features.known_fraud.value_counts().reindex([False,True],fill_value=0)
if (counts==0).any(): raise ValueError(f'Both classes are required; counts={counts.to_dict()}')

comparison=features.groupby('known_fraud',observed=False)[traffic_columns].median().reindex([False,True]).T
comparison.columns=['known_legitimate_median','known_fraudulent_median']
comparison['difference']=comparison.known_fraudulent_median-comparison.known_legitimate_median
comparison.to_csv(OUTPUT/'traffic_group_median_comparison.csv')

correlations=features[traffic_columns].corr(method='spearman')
correlations.to_csv(OUTPUT/'traffic_correlations_spearman.csv')
comparison

,known_legitimate_median,known_fraudulent_median,difference
activity_count,2.000000,3.000000,1.00000
outgoing_activity_count,0.000000,3.000000,3.00000
incoming_activity_count,1.000000,0.000000,-1.00000
outgoing_share,0.000000,1.000000,1.00000
incoming_share,0.600000,0.000000,-0.60000
total_duration,53.000000,209.000000,156.00000
duration_per_activity,28.666667,62.666667,34.00000
unique_contacts,1.000000,3.000000,2.00000
contact_per_active_hour,1.000000,1.000000,0.00000
international_activity_count,0.000000,0.000000,0.00000


In [4]:
# Robust outlier evidence: how far each SIM is from the legitimate population median/IQR.
legit=features.loc[~features.known_fraud,traffic_columns]
q1=legit.quantile(.25,numeric_only=True); q3=legit.quantile(.75,numeric_only=True); iqr=q3-q1
rows=[]
for _,row in features.iterrows():
    for col in traffic_columns:
        value=row[col]
        if pd.isna(value) or pd.isna(iqr[col]) or iqr[col]==0: continue
        direction='high' if value>q3[col] else 'low' if value<q1[col] else 'within_iqr'
        if direction!='within_iqr': rows.append({'sim_number':row.sim_number,'known_fraud':row.known_fraud,'feature':col,'value':value,'legitimate_q1':q1[col],'legitimate_q3':q3[col],'iqr_deviation':(value-q3[col])/iqr[col] if direction=='high' else (q1[col]-value)/iqr[col],'direction':direction})
evidence=pd.DataFrame(rows).sort_values('iqr_deviation',ascending=False)
evidence.to_csv(OUTPUT/'traffic_iqr_deviations.csv',index=False)
print('evidence rows:',len(evidence)); print(evidence.head(20))

evidence rows: 16240
         sim_number  known_fraud                    feature        value  \
4657   +24400530399        False             activity_count  5152.000000   
4662   +24400530399        False  activity_population_share     0.052089   
4565   +24400529818        False  activity_population_share     0.048459   
4562   +24400529818        False             activity_count  4793.000000   
14045  +24400595739        False             activity_count  3644.000000   
14048  +24400595739        False  activity_population_share     0.036842   
6517   +24400543082        False             activity_count  3144.000000   
6520   +24400543082        False  activity_population_share     0.031787   
12849  +24400587177        False             activity_count  2947.000000   
12852  +24400587177        False  activity_population_share     0.029795   
480    +24400501318        False             activity_count  2752.000000   
483    +24400501318        False  activity_population_share     0.0

Interpret traffic differences as investigative evidence only. A high IQR deviation is a reason to inspect a SIM, not proof of fraud.